In [19]:
import atmosphericRadiationDoseAndFlux as ARDF
from atmosphericRadiationDoseAndFlux import particle, particleResponse, units
import numpy as np

In [20]:
import atmosphericRadiationDoseAndFlux as ARDF
from atmosphericRadiationDoseAndFlux import particle, particleResponse, units

particleForCalculations = particle.Particle("proton")
doseType = "edose"

doseResponseForParticle = particleResponse.fullDoseResponseDict[doseType](particleForCalculations, doseType)

altitude = units.Distance(12.192 * 1000.0)

In [21]:
doseResponseForParticle.getPathToResponseFile()

'/home/xisacross/programming/DoseAndFluxCalculator/atmosphericRadiationDoseAndFlux/data/proton/edose.rpf'

In [22]:
particleResponseArray = np.genfromtxt(doseResponseForParticle.getPathToResponseFile())
particleResponseArray

array([[0.00e+00, 0.00e+00, 0.00e+00, ..., 1.56e+02, 2.16e+02, 2.92e+02],
       [0.00e+00, 0.00e+00, 0.00e+00, ..., 1.57e+02, 2.18e+02, 2.93e+02],
       [0.00e+00, 0.00e+00, 0.00e+00, ..., 1.59e+02, 2.20e+02, 2.97e+02],
       ...,
       [9.33e-05, 2.44e-04, 6.08e-04, ..., 5.17e+02, 7.92e+02, 8.51e+02],
       [1.86e-04, 4.57e-04, 1.06e-01, ..., 3.45e+02, 4.83e+02, 5.23e+02],
       [7.02e-01, 8.80e-01, 1.10e+00, ..., 2.90e+02, 3.78e+02, 4.27e+02]],
      shape=(137, 50))

In [23]:
from typing import Tuple

def calculate_altitude_layer_params(altitude_meters: float, altitude_km: float) -> Tuple[int, float]:
    """
    Calculate altitude layer index and interpolation factor using Numba optimization.
    
    Parameters
    ----------
    altitude_meters : float
        Altitude in meters
    altitude_km : float
        Altitude in kilometers
        
    Returns
    -------
    Tuple[int, float]
        Tuple containing:
        - Altitude layer index
        - Interpolation factor (f1)
        
    Raises
    ------
    Exception
        If altitude is greater than 100 km
    """
    if altitude_km > 100.0:
        # Can't use exceptions in njit, so return invalid values that will be caught
        return -1, -1.0

    if altitude_km < 0.025:
        layer = 1
        f1 = 1.0
    elif altitude_km < 1.025:
        layer = int((int(altitude_meters)-25)//50) + 1
        hr = (int(altitude_meters)-25) % 50
        f1 = 1.0 - hr/50.0
    elif altitude_km < 1.15:
        layer = 21
        f1 = 1.0 - (int(altitude_meters)-1025)/1025.0
    elif altitude_km < 5.05:
        layer = int((int(altitude_meters)-1150)//100) + 22
        hr = (int(altitude_meters)-1150) % 100
        f1 = 1.0 - hr/100.0
    elif altitude_km < 5.3:
        layer = 61
        f1 = 1.0 - (altitude_km-5.05)/0.25
    elif altitude_km < 15.1:
        layer = int((int(altitude_meters)-5300)//200) + 62
        hr = (int(altitude_meters)-5300) % 200
        f1 = 1.0 - hr/200.0
    elif altitude_km < 16.5:
        layer = 111
        f1 = 1.0 - (altitude_km-15.1)/1.4
    elif altitude_km < 38.5:
        layer = int((int(altitude_meters)-16500)//1000) + 112
        hr = (int(altitude_meters)-16500) % 1000
        f1 = 1.0 - hr/1000.0
    elif altitude_km < 40.5:
        layer = 134
        f1 = 1.0 - (altitude_km-38.5)/2.0
    elif altitude_km < 62.5:
        layer = 135
        f1 = 1.0 - (altitude_km-40.5)/22.0
    elif altitude_km < 97.5:
        layer = 136
        f1 = 1.0 - (altitude_km-40.5)/57.5
    else:
        layer = 137
        f1 = 1.0

    # Convert from 1-indexed to 0-indexed for Python arrays
    altitude_layer_index = layer - 1
    
    return altitude_layer_index, f1

def getAltitudeResponseLayer(altitude):
        """
        Calculate altitude response layer and interpolation factor.
        
        Parameters
        ----------
        altitude : Distance
            Altitude object
            
        Raises
        ------
        Exception
            If altitude is greater than 100 km
        """
        # Use Numba-optimized function for layer calculation
        altitude_layer_index, f1 = calculate_altitude_layer_params(
            altitude.meters, altitude.km
        )
        
        # Check for error from Numba function (altitude > 100 km)
        if altitude_layer_index == -1:
            raise Exception("altitude.km has to be less than 100 km!")

        return altitude_layer_index, f1

In [24]:
altIndex, f1 = getAltitudeResponseLayer(altitude)
altIndexAbove = altIndex + 1

print("altIndex, f1, altIndexAbove: ", altIndex, f1, altIndexAbove)

altIndex, f1, altIndexAbove:  95 0.54 96


In [25]:
def calculate_weighted_fluxes(energy_bin_edges: np.ndarray, flux_values: np.ndarray, 
                             cutoff_energy: float) -> Tuple[np.ndarray, int]:
    """
    Calculate weighted flux values based on energy cutoff using Numba optimization.
    
    Parameters
    ----------
    energy_bin_edges : np.ndarray
        Energy bin edges
    flux_values : np.ndarray
        Flux values
    cutoff_energy : float
        Cutoff energy in MeV
        
    Returns
    -------
    Tuple[np.ndarray, int]
        Tuple containing:
        - Weighted flux values
        - Energy index for cutoff
    """
    # Make a copy of flux_values to avoid modifying the original
    weighted_fluxes = flux_values.copy()
    
    # Find energy index for cutoff
    cutoff_bin_index = 0
    while cutoff_bin_index < len(energy_bin_edges) and cutoff_energy > energy_bin_edges[cutoff_bin_index]:
        cutoff_bin_index += 1
    
    # Apply weighting based on cutoff energy
    if cutoff_bin_index > 0:
        cutoff_bin_index = cutoff_bin_index - 1
        
        energy_delta = energy_bin_edges[cutoff_bin_index+1] - cutoff_energy
        total_bin_width = energy_bin_edges[cutoff_bin_index+1] - energy_bin_edges[cutoff_bin_index]
        
        fraction_in_bin = energy_delta / total_bin_width
        weighted_fluxes[cutoff_bin_index] = weighted_fluxes[cutoff_bin_index] * fraction_in_bin
    
    return weighted_fluxes, cutoff_bin_index

In [26]:
def getDoseResponseTerms(particleResponseArray, altitudeLayerIndex: int, altIndexAbove: int, 
                        energyIndex: int, f1: float) -> Tuple[float, float]:
    """
    Get dose response terms for calculation.
    
    Parameters
    ----------
    altitudeLayerIndex : int
        Index for altitude layer
    altIndexAbove : int
        Index for altitude layer above
    energyIndex : int
        Index for energy bin
    f1 : float
        Interpolation factor
        
    Returns
    -------
    Tuple[float, float]
        First and second dose response terms
    """
    firstDoseResponseTerm = particleResponseArray[altitudeLayerIndex, energyIndex] * f1
    secondDoseResponseTerm = particleResponseArray[altIndexAbove, energyIndex] * (f1 - 1)

    return (firstDoseResponseTerm, secondDoseResponseTerm)

In [27]:
def dose_response_term(response_values: np.ndarray, 
                       alt_index: int, 
                       alt_index_above: int, 
                       energy_idx: int, 
                       f1: float) -> float:
    """
    Compute the difference between the interpolated response terms for a given energy bin.
    """
    first_term = response_values[alt_index, energy_idx] * f1
    second_term = response_values[alt_index_above, energy_idx] * (f1 - 1)
    return first_term - second_term

In [28]:
import numpy as np

In [29]:
full_energy_bins_array = 10**(0.1*(np.array(range(1,52))-1)+1)

def get_energy_index_from_energy_MeV(energy_MeV: float) -> int:

   return np.argmin(np.abs(full_energy_bins_array - energy_MeV))

In [30]:
full_energy_bins_array[1]

np.float64(12.589254117941675)

In [31]:
get_energy_index_from_energy_MeV(full_energy_bins_array[0])

np.int64(0)

In [32]:
def full_calculate_response_terms(energy_in_MeV = 1000, altitude_in_km = 12.192):

    from atmosphericRadiationDoseAndFlux import particle, particleResponse, units

    particleForCalculations = particle.Particle("proton")
    doseType = "edose"

    doseResponseForParticle = particleResponse.fullDoseResponseDict[doseType](particleForCalculations, doseType)

    altitude = units.Distance(altitude_in_km * 1000.0)

    particleResponseArray = np.genfromtxt(doseResponseForParticle.getPathToResponseFile())
    
    altIndex, f1 = getAltitudeResponseLayer(altitude)
    altIndexAbove = altIndex + 1

    energy_idx = get_energy_index_from_energy_MeV(energy_in_MeV)

    return dose_response_term(response_values=particleResponseArray, 
                        alt_index=altIndex, 
                        alt_index_above=altIndexAbove, 
                        energy_idx=energy_idx, 
                        f1=f1)

In [40]:
full_calculate_response_terms(energy_in_MeV = 4000, altitude_in_km = 12.192)

np.float64(12.584)

In [34]:
def calculate_dose_response(weighted_fluxes: np.ndarray, 
                           response_values: np.ndarray, 
                           alt_index: int, 
                           alt_index_above: int, 
                           f1: float) -> float:
    """
    Calculate dose response using Numba optimization.
    
    Parameters
    ----------
    weighted_fluxes : np.ndarray
        Array of weighted flux values
    response_values : np.ndarray
        Array of response values
    alt_index : int
        Index for altitude layer
    alt_index_above : int
        Index for altitude layer above
    f1 : float
        Interpolation factor
        
    Returns
    -------
    float
        Calculated dose value
    """
    output_dose = 0.0
    for energy_idx in range(50):
        term = dose_response_term(response_values, alt_index, alt_index_above, energy_idx, f1)
        output_dose += weighted_fluxes[energy_idx] * term
    return output_dose

In [35]:
import pandas as pd

In [36]:
pd.DataFrame(particleResponseArray)

,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
0,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0000,0.0000,0.0000,0.000,...,3.78,9.53,18.4,31.0,49.6,74.6,109.0,156.0,216.0,292.0
1,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0000,0.0000,0.0000,0.000,...,3.79,9.62,18.4,31.1,49.9,75.2,109.0,157.0,218.0,293.0
2,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0000,0.0000,0.0000,0.000,...,3.82,9.86,18.8,31.3,50.4,76.0,111.0,159.0,220.0,297.0
3,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0000,0.0000,0.0000,0.000,...,3.96,9.76,18.9,31.8,51.0,76.8,111.0,161.0,222.0,300.0
4,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.0000,0.0000,0.0000,0.000,...,3.95,9.86,19.0,32.0,51.2,77.4,113.0,161.0,225.0,304.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,0.000082,0.000222,0.000536,0.00129,0.00327,0.00714,0.0147,0.0285,0.0530,0.764,...,85.50,122.00,158.0,223.0,291.0,369.0,482.0,624.0,935.0,1130.0
133,0.000085,0.000226,0.000566,0.00135,0.00342,0.00744,0.0153,0.0294,0.0576,1.450,...,84.00,120.00,158.0,232.0,277.0,342.0,469.0,595.0,932.0,1010.0
134,0.000093,0.000244,0.000608,0.00148,0.00370,0.00819,0.0165,0.0323,0.3320,3.240,...,83.20,117.00,146.0,204.0,252.0,314.0,423.0,517.0,792.0,851.0
135,0.000186,0.000457,0.106000,0.36100,0.77000,1.35000,3.9500,7.8600,12.0000,16.900,...,76.00,94.10,114.0,151.0,168.0,211.0,266.0,345.0,483.0,523.0


In [37]:
doseResponseForParticle